In [1]:
!pip -q install fastapi uvicorn python-multipart pillow requests opencv-python-headless gdown
!pip -q install 'git+https://github.com/facebookresearch/sam3.git'


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 195.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sam3 0.1.0 requires numpy<2,>=1.26, but you have numpy 2.4.4 which is incompatible.
numba 0.60.0 requires numpy<2.1,>=1.22, but you have numpy 2.4.4 which is incompatible.
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
jax 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
cupy-cuda12x 14.0.1 requires numpy<2.6,>=2.0, but you have numpy 1.26.4 which is incompatible.
jaxlib 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
raste

# This is the common **first-run** NumPy/torchvision error:

# `ValueError: numpy.dtype size changed, may indicate binary incompatibility`

# **If that error appears, restart the session and run all cells again**

In [2]:
import torch
import torchvision
print('PyTorch version:', torch.__version__)
print('Torchvision version:', torchvision.__version__)
print('CUDA is available:', torch.cuda.is_available())
!nvidia-smi


PyTorch version: 2.10.0+cu128
Torchvision version: 0.25.0+cu128
CUDA is available: True
Sat May  9 19:28:58 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX PRO 6000 Blac...    Off |   00000000:05:00.0 Off |                    0 |
| N/A   29C    P0             48W /  600W |       3MiB /  97887MiB |      0%      Default |
|                                         |         

In [3]:
from pathlib import Path
import gdown

# SAM checkpoint file is stored on Google Drive:
CHECKPOINT_URL = "https://drive.google.com/file/d/17ftyxnIUuabLsjzQYO3-YmXTcNPnKcup/view?usp=sharing"

LOCAL_CHECKPOINT_PATH = Path("/content/sam3_video_predictor.pt")
MIN_CHECKPOINT_BYTES = 2 * 1024 ** 3

if not LOCAL_CHECKPOINT_PATH.exists() or LOCAL_CHECKPOINT_PATH.stat().st_size < MIN_CHECKPOINT_BYTES:
    if LOCAL_CHECKPOINT_PATH.exists():
        LOCAL_CHECKPOINT_PATH.unlink()
    print("Downloading SAM3 checkpoint from Google Drive...")
    gdown.download(url=CHECKPOINT_URL, output=str(LOCAL_CHECKPOINT_PATH), quiet=False, fuzzy=True)

if not LOCAL_CHECKPOINT_PATH.is_file():
    raise FileNotFoundError(f"Checkpoint download failed: {LOCAL_CHECKPOINT_PATH}")

if LOCAL_CHECKPOINT_PATH.stat().st_size < MIN_CHECKPOINT_BYTES:
    raise RuntimeError(f"Downloaded file is too small to be the checkpoint: {LOCAL_CHECKPOINT_PATH.stat().st_size} bytes")

print()
print(LOCAL_CHECKPOINT_PATH)
print(f"Size: {LOCAL_CHECKPOINT_PATH.stat().st_size / (1024 ** 3):.2f} GB")


Downloading...
From (original): https://drive.google.com/uc?id=17ftyxnIUuabLsjzQYO3-YmXTcNPnKcup
From (redirected): https://drive.google.com/uc?id=17ftyxnIUuabLsjzQYO3-YmXTcNPnKcup&confirm=t&uuid=ded2d6f5-de74-4d71-a8a4-4b88526af30c
To: /content/sam3_video_predictor.pt
100%|██████████| 3.45G/3.45G [00:36<00:00, 93.4MB/s]


/content/sam3_video_predictor.pt
Size: 3.21 GB


In [ ]:
%%writefile sam_server.py
from contextlib import nullcontext
from dataclasses import dataclass
from pathlib import Path
from threading import Lock
import io
import time

from fastapi import FastAPI, File, Form, UploadFile
from PIL import Image
import torch

from sam3.model_builder import build_sam3_image_model
from sam3.model.sam3_image_processor import Sam3Processor

app = FastAPI()
device = "cuda" if torch.cuda.is_available() else "cpu"
LOCAL_CHECKPOINT_PATH = Path("/content/sam3_video_predictor.pt")


def build_local_sam3_image_model():
    if not LOCAL_CHECKPOINT_PATH.is_file():
        raise FileNotFoundError(f"SAM3 checkpoint not found: {LOCAL_CHECKPOINT_PATH}")

    print("Loading SAM3 image model from checkpoint", flush=True)
    print(f"  {LOCAL_CHECKPOINT_PATH}", flush=True)

    try:
        model = build_sam3_image_model(
            device=device,
            checkpoint_path=str(LOCAL_CHECKPOINT_PATH),
            load_from_HF=False,
        )
    except TypeError:
        model = build_sam3_image_model(
            checkpoint_path=str(LOCAL_CHECKPOINT_PATH),
            load_from_HF=False,
        )

    return model, str(LOCAL_CHECKPOINT_PATH)


model, sam3_checkpoint_path = build_local_sam3_image_model()
try:
    processor = Sam3Processor(model, device=device)
except TypeError:
    processor = Sam3Processor(model)
lock = Lock()


@dataclass
class Track:
    track_id: int
    box: list
    score: float
    age: int
    hits: int
    misses: int
    frame_index: int


class BoxTracker:
    def __init__(self):
        self.tracks = {}
        self.next_id = 1
        self.frame_index = 0

    def reset(self):
        self.tracks = {}
        self.next_id = 1
        self.frame_index = 0

    def iou(self, a, b):
        ax1, ay1, ax2, ay2 = a
        bx1, by1, bx2, by2 = b
        ix1 = max(ax1, bx1)
        iy1 = max(ay1, by1)
        ix2 = min(ax2, bx2)
        iy2 = min(ay2, by2)
        iw = max(0.0, ix2 - ix1)
        ih = max(0.0, iy2 - iy1)
        inter = iw * ih
        aa = max(0.0, ax2 - ax1) * max(0.0, ay2 - ay1)
        ba = max(0.0, bx2 - bx1) * max(0.0, by2 - by1)
        union = aa + ba - inter
        if union <= 0.0:
            return 0.0
        return inter / union

    def update(self, detections, track_iou, max_misses):
        self.frame_index += 1
        track_ids = list(self.tracks.keys())
        pairs = []

        for ti in track_ids:
            for di, det in enumerate(detections):
                pairs.append((self.iou(self.tracks[ti].box, det["box"]), ti, di))

        pairs.sort(reverse=True, key=lambda x: x[0])
        used_tracks = set()
        used_detections = set()

        for value, ti, di in pairs:
            if value < track_iou:
                break
            if ti in used_tracks or di in used_detections:
                continue

            det = detections[di]
            track = self.tracks[ti]
            track.box = det["box"]
            track.score = det["score"]
            track.age += 1
            track.hits += 1
            track.misses = 0
            track.frame_index = self.frame_index
            used_tracks.add(ti)
            used_detections.add(di)

        for ti in track_ids:
            if ti not in used_tracks:
                self.tracks[ti].age += 1
                self.tracks[ti].misses += 1

        for di, det in enumerate(detections):
            if di in used_detections:
                continue
            ti = self.next_id
            self.next_id += 1
            self.tracks[ti] = Track(ti, det["box"], det["score"], 1, 1, 0, self.frame_index)

        remove = [ti for ti, track in self.tracks.items() if track.misses > max_misses]
        for ti in remove:
            del self.tracks[ti]

        visible = [track for track in self.tracks.values() if track.misses == 0]
        visible.sort(key=lambda t: t.track_id)

        return [
            {
                "track_id": t.track_id,
                "box": t.box,
                "score": t.score,
                "age": t.age,
                "hits": t.hits,
                "misses": t.misses,
            }
            for t in visible
        ]


tracker = BoxTracker()


def as_list(value):
    if value is None:
        return []
    if hasattr(value, "detach"):
        value = value.detach().cpu()
    if hasattr(value, "tolist"):
        return value.tolist()
    return value


def get_output_value(output, names):
    for name in names:
        if isinstance(output, dict) and name in output:
            return output[name]
        if hasattr(output, name):
            return getattr(output, name)
    return None


def normalize_boxes(boxes, width, height):
    normalized = []
    for box in boxes:
        if len(box) < 4:
            continue

        x1, y1, x2, y2 = [float(v) for v in box[:4]]

        if max(abs(x1), abs(y1), abs(x2), abs(y2)) <= 1.5:
            x1 *= width
            x2 *= width
            y1 *= height
            y2 *= height

        x1 = min(max(x1, 0.0), float(width - 1))
        x2 = min(max(x2, 0.0), float(width - 1))
        y1 = min(max(y1, 0.0), float(height - 1))
        y2 = min(max(y2, 0.0), float(height - 1))

        if x2 < x1:
            x1, x2 = x2, x1
        if y2 < y1:
            y1, y2 = y2, y1
        if x2 - x1 >= 2.0 and y2 - y1 >= 2.0:
            normalized.append([x1, y1, x2, y2])

    return normalized


def iou(a, b):
    ax1, ay1, ax2, ay2 = a
    bx1, by1, bx2, by2 = b
    ix1 = max(ax1, bx1)
    iy1 = max(ay1, by1)
    ix2 = min(ax2, bx2)
    iy2 = min(ay2, by2)
    iw = max(0.0, ix2 - ix1)
    ih = max(0.0, iy2 - iy1)
    inter = iw * ih
    aa = max(0.0, ax2 - ax1) * max(0.0, ay2 - ay1)
    ba = max(0.0, bx2 - bx1) * max(0.0, by2 - by1)
    union = aa + ba - inter
    if union <= 0.0:
        return 0.0
    return inter / union


def suppress_duplicates(detections, duplicate_iou):
    ordered = sorted(detections, key=lambda d: d["score"], reverse=True)
    keep = []

    for det in ordered:
        if all(iou(det["box"], other["box"]) < duplicate_iou for other in keep):
            keep.append(det)

    return keep


@app.get("/health")
def health():
    return {
        "ok": True,
        "device": device,
        "tracks": len(tracker.tracks),
        "checkpoint_path": sam3_checkpoint_path,
        "huggingface_load": False,
    }


@app.post("/reset")
def reset():
    with lock:
        tracker.reset()
    return {"ok": True}


@app.post("/infer")
async def infer(
    file: UploadFile = File(...),
    prompt: str = Form("poster, flyer"),
    conf: float = Form(0.25),
    track_iou: float = Form(0.30),
    max_misses: int = Form(8),
    duplicate_iou: float = Form(0.65),
):
    data = await file.read()
    image = Image.open(io.BytesIO(data)).convert("RGB")
    width, height = image.size
    t0 = time.time()
    autocast_ctx = torch.autocast(device_type="cuda", dtype=torch.bfloat16) if device == "cuda" else nullcontext()

    with lock:
        with torch.inference_mode(), autocast_ctx:
            state = processor.set_image(image)
            output = processor.set_text_prompt(state=state, prompt=prompt)

        raw_boxes = as_list(get_output_value(output, ["boxes", "pred_boxes"]))
        raw_scores = as_list(get_output_value(output, ["scores", "pred_scores", "confidence", "confidences"]))
        boxes = normalize_boxes(raw_boxes, width, height)

        if not raw_scores:
            raw_scores = [1.0 for _ in boxes]

        detections = []
        for box, score in zip(boxes, raw_scores):
            score = float(score)
            if score >= conf:
                detections.append({"box": box, "score": score})

        detections = suppress_duplicates(detections, duplicate_iou)
        tracks = tracker.update(detections, track_iou, max_misses)

    server_ms = (time.time() - t0) * 1000.0

    return {
        "ok": True,
        "device": device,
        "prompt": prompt,
        "width": width,
        "height": height,
        "num_detections": len(detections),
        "tracks": tracks,
        "server_ms": server_ms,
        "checkpoint_path": sam3_checkpoint_path,
    }


Writing sam_server.py


In [5]:
# Start `sam_server.py`

%%bash
cd /content
pkill -f "uvicorn sam_server:app" || true
rm -f server.log server.pid /tmp/sam_health.json
nohup python -m uvicorn sam_server:app --host 0.0.0.0 --port 8000 > server.log 2>&1 &
echo $! > server.pid

echo "Waiting for SAM server to finish loading..."
for i in $(seq 1 300); do
    if curl -sf http://127.0.0.1:8000/health > /tmp/sam_health.json; then
        echo "SAM server is ready."
        cat /tmp/sam_health.json
        echo
        exit 0
    fi

    if ! kill -0 "$(cat server.pid)" 2>/dev/null; then
        echo "SAM server stopped before it became ready."
        tail -200 server.log
        exit 1
    fi

    if [ $((i % 15)) -eq 0 ]; then
        echo "Still loading... elapsed ${i}s"
        tail -20 server.log
    fi

    sleep 1
done

echo "Timed out waiting for SAM server."
tail -200 server.log
exit 1


Waiting for SAM server to finish loading...
SAM server is ready.
{"ok":true,"device":"cuda","tracks":0,"checkpoint_path":"/content/sam3_video_predictor.pt","huggingface_load":false}


In [6]:
!curl http://127.0.0.1:8000/health

{"ok":true,"device":"cuda","tracks":0,"checkpoint_path":"/content/sam3_video_predictor.pt","huggingface_load":false}

# ***Leave this cell running and copy the `https://...trycloudflare.com` URL into the Qt client***


In [ ]:
%cd /content
!pkill -f cloudflared || true
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared
!./cloudflared tunnel --url http://localhost:8000 --no-autoupdate


/content
^C
2026-05-09T19:32:13Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-05-09T19:32:13Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-05-09T19:32:16Z INF +--------------------------------------------------------------------------------------------+
2026-05-09T19:32:16Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-05-09T19:32:16Z INF |  https://votes-bonus-casino-moving.tryclou

In [8]:
!pkill -f "uvicorn sam_server:app" || true
!pkill -f cloudflared || true


^C
^C
